Block triangular preconditioners for the heat equation
====================================================

This demo applies the method suggested in:

Mardal, Nilssen, Staff, "Order-optimal preconditioners for implicit
Runge-Kutta schemes applied to parabolic PDEs", SISC 29(1): 361--375 (2007),

to our ongoing heat equation demonstration problem on $\Omega = [0,10]
\times [0,10]$, with boundary $\Gamma$, giving rise to the weak form

$$
(u_t, v) + (\nabla u, \nabla v) = (f, v)
$$

A multi-stage RK method applied to the heat equation gives a
block-structured system.  The on-diagonal blocks are quite similar to
what one obtains from a backward Euler discretization of the equation.

With a 2-stage method, we have

$$
\left[ \begin{array}{cc}
M + a_{11} \Delta t K &  a_{12} \Delta t K \\
a_{21} \Delta t K & M + a_{22} \Delta t K
\end{array}
\right]
\left[ \begin{array}{c} k_1 \\ k_2 \end{array} \right]
   =
\left[ \begin{array}{cc} A_{11} & A_{12} \\ A_{21} & A_{22} \end{array} \right]
   \left[ \begin{array}{c} k_1 \\ k_2 \end{array} \right]
   = \left[ \begin{array}{c} f_1 \\ f_2 \end{array} \right]
$$



And the suggestion (analyzed rigorously) of Mardal, Nilssen, and Staff
is to use a block diagonal preconditioner:

$$
P = \left[ \begin{array}{cc} A_{11} & 0 \\ 0 & A_{22} \end{array} \right]
$$


This allows one to leverage an existing methodology for a low order
method like backward Euler for the diagonal blocks.  In our case, we
will simply use an algebraic multigrid scheme, although one could
certainly use geometric multigrid or some other technique.

In [2]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

In [2]:
from firedrake import *  # noqa: F403
from irksome import LobattoIIIC, TimeStepper, Dt

Here is our setup as in the previous demo:

In [8]:
butcher_tableau = LobattoIIIC(2)
N = 64
x0 = 0.0
x1 = 10.0
y0 = 0.0
y1 = 10.0
msh = RectangleMesh(N, N, x1, y1)
dt = Constant(10. / N)
t = Constant(0.0)
V = FunctionSpace(msh, "CG", 1)
x, y = SpatialCoordinate(msh)
S = Constant(2.0)
C = Constant(1000.0)
B = (x-Constant(x0))*(x-Constant(x1))*(y-Constant(y0))*(y-Constant(y1))/C
R = (x * x + y * y) ** 0.5
uexact = B * atan(t)*(pi / 2.0 - atan(S * (R - t)))
rhs = Dt(uexact) - div(grad(uexact))
u = Function(V)
u.interpolate(uexact)
v = TestFunction(V)
F = inner(Dt(u), v)*dx + inner(grad(u), grad(v))*dx - inner(rhs, v)*dx
bc = DirichletBC(V, 0, "on_boundary")

Now, we define the solver parameters.  PETSc-speak for taking the
block diagonal is an "additive fieldsplit", and we specify just using PETSc's algebraic multigrid on all of the blocks

In [3]:
params = {
    "mat_type": "aij",
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "ksp_monitor": None,
    "pc_type": "fieldsplit",   # block preconditioner
    "pc_fieldsplit_type": "additive",  # block diagaonal
    "fieldsplit": {
    "ksp_type": "preonly",
        "pc_type": "gamg"
    }
}

Note that we have used the same technique for each RK stage, which is
probably typical.  However, it is not necessary at all.
     
To test this preconditioning strategy, we'll create a time stepping
object which will set up the variational problem for us::

In [9]:
stepper = TimeStepper(F, butcher_tableau, t, dt, u, bcs=bc,
                      solver_parameters=params)

Since we're just testing the efficacy of the preconditioner, we'll just take one step:

In [10]:
stepper.advance()

tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)


    Residual norms for firedrake_5_ solve.
    0 KSP Residual norm 2.370167625025e+00
    1 KSP Residual norm 7.521537865467e-02
    2 KSP Residual norm 2.449435553322e-02
    3 KSP Residual norm 1.404113546811e-02
    4 KSP Residual norm 5.165488219455e-03
    5 KSP Residual norm 2.631609621099e-03
    6 KSP Residual norm 1.204298130962e-03
    7 KSP Residual norm 5.764952188047e-04
    8 KSP Residual norm 2.708319036689e-04
    9 KSP Residual norm 1.309874413186e-04
   10 KSP Residual norm 6.399328352214e-05
   11 KSP Residual norm 3.050203812599e-05
   12 KSP Residual norm 1.524883369093e-05


However, this preconditioner does not scale with the number of stages:

In [7]:
stepper = TimeStepper(F, LobattoIIIC(5), t, dt, u, bcs=bc,
                      solver_parameters=params)
stepper.advance()

tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 35 more than tenfold greater than any argument/coefficient degree (max 1)


    Residual norms for firedrake_3_ solve.
    0 KSP Residual norm 3.835714508232e+00
    1 KSP Residual norm 1.551104204490e-01
    2 KSP Residual norm 6.571858874389e-02
    3 KSP Residual norm 4.207522210473e-02
    4 KSP Residual norm 2.042372129315e-02
    5 KSP Residual norm 1.367336804187e-02
    6 KSP Residual norm 7.304103908138e-03
    7 KSP Residual norm 4.901198999741e-03
    8 KSP Residual norm 3.922662014658e-03
    9 KSP Residual norm 3.034220786532e-03
   10 KSP Residual norm 2.241859497982e-03
   11 KSP Residual norm 1.719818632962e-03
   12 KSP Residual norm 1.500389835635e-03
   13 KSP Residual norm 1.355258882427e-03
   14 KSP Residual norm 9.859935506635e-04
   15 KSP Residual norm 6.831180821491e-04
   16 KSP Residual norm 5.819352390924e-04
   17 KSP Residual norm 4.496716979066e-04
   18 KSP Residual norm 3.462944950276e-04
   19 KSP Residual norm 2.902736797947e-04
   20 KSP Residual norm 2.176137322007e-04
   21 KSP Residual norm 1.917431604266e-04
   22 KSP R

Rana *et al* (SISC 2021) propose a different approach to block triangular preconditioning that overcomes this difficulty.  It rediscretizes the time-dependent problem with a block-triangular approximation to the Butcher matrix, which leads to a block triangular system.  One approach is to take the standard factorization $A = LDU$, and then define
$$
\tilde{A} = LD.
$$

This is implemented quite generally in Irksome as Python preconditioner (built on top of the auxiliary operator framework).  Here, we make the Rana approximation, then do a multiplicative fieldsplit and approximate the inverses of the diagonal blocks with a sweep of AMG:

In [13]:
ranaparams = {
    "mat_type": "aij",
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "ksp_monitor": None,
    "pc_type": "python",
    "pc_python_type": "irksome.RanaLD",
    "aux": {
        "pc_type": "fieldsplit",
        "pc_fieldsplit_type": "multiplicative",
        "fieldsplit": {
            "ksp_type": "preonly",
            "pc_type": "gamg"
        }
    }
}

stepper = TimeStepper(F, LobattoIIIC(5), t, dt, u, bcs=bc,
                      solver_parameters=ranaparams)
stepper.advance()


    Residual norms for firedrake_8_ solve.
    0 KSP Residual norm 3.311214795517e+00
    1 KSP Residual norm 6.955420630200e-02
    2 KSP Residual norm 1.667412153267e-02
    3 KSP Residual norm 5.985389868126e-03
    4 KSP Residual norm 1.629113064343e-03
    5 KSP Residual norm 8.702444076755e-04
    6 KSP Residual norm 5.431744096727e-04
    7 KSP Residual norm 3.333310451091e-04
    8 KSP Residual norm 1.679454773821e-04
    9 KSP Residual norm 7.535800947522e-05
   10 KSP Residual norm 3.352110119525e-05
   11 KSP Residual norm 1.777650697319e-05


It is important to note that the cost of applying the Rana preconditioner per iteration is essentially the same as the original multiplicative field split, but we require many fewer outer iterations.
